In [1]:
import pandas as pd
import numpy as np
from scipy import stats


In [3]:
blue = pd.read_excel("EE survey blue responses.xlsx")
red = pd.read_excel("EE red survey (Responses).xlsx")



In [4]:
brand_columns = [
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [Audi]",
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [BMW]",
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [Honda]",
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [Hyundai]",
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [Jaguar Land Rover]",
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [Mahindra]",
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [Mercedes Benz]",
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [Mini Cooper]",
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [Morris Garages (MG)]",
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [Skoda]",
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [Suzuki]",
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [Tata]",
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [Toyota]",
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [Volvo]",
    "Rate these car companies based on safety. (5 is the best and 1 is the worst) [Volkswagen]"
]



In [6]:
def compute_brand_summary(df, label):

    # Force all brand columns to numeric safely
    for col in brand_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    summary = pd.DataFrame()
    summary["Brand"] = [col.split("[")[-1].replace("]", "") for col in brand_columns]
    summary[f"{label}_Mean"] = df[brand_columns].mean(skipna=True).values
    summary[f"{label}_SD"] = df[brand_columns].std(skipna=True).values
    summary[f"{label}_Count"] = df[brand_columns].count().values

    return summary


In [7]:
blue_summary = compute_brand_summary(blue.copy(), "Blue")
red_summary = compute_brand_summary(red.copy(), "Red")

brand_summary = pd.merge(blue_summary, red_summary, on="Brand")
brand_summary.to_excel("01_Brand_Level_Summary.xlsx", index=False)

print("Section A fixed and exported.")


Section A fixed and exported.


In [8]:
# ===============================
# SECTION B — WELCH T TEST
# ===============================

# Ensure numeric conversion again (safety columns)
for col in brand_columns:
    blue[col] = pd.to_numeric(blue[col], errors='coerce')
    red[col] = pd.to_numeric(red[col], errors='coerce')

# Flatten into 1D arrays
blue_all = blue[brand_columns].values.flatten()
red_all = red[brand_columns].values.flatten()

# Remove NaNs
blue_all = blue_all[~np.isnan(blue_all)]
red_all = red_all[~np.isnan(red_all)]

# Compute test
t_stat, p_val = stats.ttest_ind(blue_all, red_all, equal_var=False)

t_results = pd.DataFrame({
    "Blue_Mean":[blue_all.mean()],
    "Red_Mean":[red_all.mean()],
    "Blue_SD":[blue_all.std(ddof=1)],
    "Red_SD":[red_all.std(ddof=1)],
    "Blue_N":[len(blue_all)],
    "Red_N":[len(red_all)],
    "t_statistic":[t_stat],
    "p_value":[p_val]
})

t_results.to_excel("02_T_Test_Results.xlsx", index=False)

print("Section B complete.")


Section B complete.


In [9]:
# ===============================
# SECTION C — CHI SQUARE TEST
# ===============================

def get_frequency(df):
    ratings = df[brand_columns].values.flatten()
    ratings = ratings[~np.isnan(ratings)]
    freq = pd.Series(ratings).value_counts().sort_index()
    return freq

blue_freq = get_frequency(blue)
red_freq = get_frequency(red)

# Ensure all ratings 1–5 appear
all_ratings = [1,2,3,4,5]

blue_freq = blue_freq.reindex(all_ratings, fill_value=0)
red_freq = red_freq.reindex(all_ratings, fill_value=0)

chi_table = pd.DataFrame({
    "Rating":all_ratings,
    "Blue_Frequency":blue_freq.values,
    "Red_Frequency":red_freq.values
})

# Perform test
contingency = chi_table[["Blue_Frequency","Red_Frequency"]].values
chi_stat, chi_p, dof, expected = stats.chi2_contingency(contingency)

chi_results = pd.DataFrame({
    "Chi_Square":[chi_stat],
    "Degrees_of_Freedom":[dof],
    "p_value":[chi_p]
})

chi_table.to_excel("03_Chi_Frequency_Table.xlsx", index=False)
chi_results.to_excel("04_Chi_Square_Results.xlsx", index=False)

print("Section C complete.")


Section C complete.


In [10]:
price_col = "What was the purchase price?"
fair_price_col = "What price range would you consider fair for this car?"


In [11]:
# ===============================
# SECTION D — PPBDAI
# ===============================

def compute_ppbdai(df):

    # Convert to numeric
    df[price_col] = pd.to_numeric(df[price_col], errors='coerce')
    df[fair_price_col] = pd.to_numeric(df[fair_price_col], errors='coerce')

    df_clean = df[[price_col, fair_price_col]].dropna()

    df_clean["Abs_Deviation"] = abs(df_clean[fair_price_col] - df_clean[price_col])

    return df_clean["Abs_Deviation"].mean(), df_clean

PPBDAI_Blue, blue_dev_table = compute_ppbdai(blue.copy())
PPBDAI_Red, red_dev_table = compute_ppbdai(red.copy())

ppbdai_results = pd.DataFrame({
    "PPBDAI_Blue":[PPBDAI_Blue],
    "PPBDAI_Red":[PPBDAI_Red],
    "Absolute_Contraction":[PPBDAI_Blue - PPBDAI_Red],
    "Percent_Contraction":[((PPBDAI_Blue - PPBDAI_Red)/PPBDAI_Blue)*100]
})

ppbdai_results.to_excel("05_PPBDAI_Results.xlsx", index=False)
blue_dev_table.to_excel("05A_Blue_PPBDI_Row_Level.xlsx", index=False)
red_dev_table.to_excel("05B_Red_PPBDI_Row_Level.xlsx", index=False)

print("Section D complete.")


Section D complete.


In [12]:
# ===============================
# SECTION E — MAD CALCULATION
# ===============================

# INSERT YOUR REAL BENCHMARK VALUES HERE
expert_benchmark = {
    "Audi":3.40,
    "BMW":3.45,
    "Honda":3.30,
    "Hyundai":3.20,
    "Jaguar Land Rover":3.60,
    "Mahindra":3.10,
    "Mercedes Benz":3.50,
    "Mini Cooper":3.30,
    "Morris Garages (MG)":3.20,
    "Skoda":3.35,
    "Suzuki":2.80,
    "Tata":3.50,
    "Toyota":3.60,
    "Volvo":4.20,
    "Volkswagen":3.55
}

brand_summary["Benchmark"] = brand_summary["Brand"].map(expert_benchmark)

brand_summary["Blue_Abs_Dev"] = abs(brand_summary["Blue_Mean"] - brand_summary["Benchmark"])
brand_summary["Red_Abs_Dev"] = abs(brand_summary["Red_Mean"] - brand_summary["Benchmark"])

MAD_Blue = brand_summary["Blue_Abs_Dev"].mean()
MAD_Red = brand_summary["Red_Abs_Dev"].mean()

MAD_results = pd.DataFrame({
    "MAD_Blue":[MAD_Blue],
    "MAD_Red":[MAD_Red],
    "Absolute_Change":[MAD_Blue - MAD_Red],
    "Percent_Reduction":[((MAD_Blue - MAD_Red)/MAD_Blue)*100]
})

brand_summary.to_excel("06_Brand_with_Benchmark.xlsx", index=False)
MAD_results.to_excel("06_MAD_Results.xlsx", index=False)

print("Section E complete.")


Section E complete.


In [13]:
# ===============================
# SECTION F — TRANSMISSION TABLE
# ===============================

comparison = pd.DataFrame({
    "Metric":["Mean_Safety","MAD","PPBDAI"],
    "Blue":[blue_all.mean(), MAD_Blue, PPBDAI_Blue],
    "Red":[red_all.mean(), MAD_Red, PPBDAI_Red]
})

comparison["Absolute_Change"] = comparison["Blue"] - comparison["Red"]
comparison["Percent_Change"] = (comparison["Absolute_Change"]/comparison["Blue"])*100

comparison.to_excel("07_Comparative_Transmission.xlsx", index=False)

print("Section F complete.")


Section F complete.
